#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#### Loading Prepared Datasets

In [2]:
data_dir = Path("../data/processed")

train_df = pd.read_parquet(data_dir / "daily_revenue_train.parquet")
test_df = pd.read_parquet(data_dir / "daily_revenue_test.parquet")
calendar_df = pd.read_parquet(data_dir / "trading_calendar.parquet")

train_df = train_df.sort_values("Transaction_Date").reset_index(drop=True)
test_df = test_df.sort_values("Transaction_Date").reset_index(drop=True)
calendar_df = calendar_df.sort_values("Transaction_Date").reset_index(drop=True)

assert train_df["Transaction_Date"].max() < test_df["Transaction_Date"].min()
assert train_df["Transaction_Date"].is_unique
assert test_df["Transaction_Date"].is_unique

print("Training Dataset Shape:", train_df.shape)
print("Testing Dataset Shape:", test_df.shape)
print("Trading Calendar Shape:", calendar_df.shape)

print("\nTraining Date Range:")
print(train_df["Transaction_Date"].min(), "to", train_df["Transaction_Date"].max())

print("\nTesting Date Range:")
print(test_df["Transaction_Date"].min(), "to", test_df["Transaction_Date"].max())

print("\nTraining Dataset Preview:")
display(train_df.head())

print("\nTesting Dataset Preview:")
display(test_df.head())

print("\nTrading Calendar Preview:")
display(calendar_df.head())

Training Dataset Shape: (244, 2)
Testing Dataset Shape: (61, 2)
Trading Calendar Shape: (374, 5)

Training Date Range:
2010-12-01 00:00:00 to 2011-09-29 00:00:00

Testing Date Range:
2011-09-30 00:00:00 to 2011-12-09 00:00:00

Training Dataset Preview:


,Transaction_Date,Daily_Revenue
0,2010-12-01,57328.60
1,2010-12-02,46174.28
2,2010-12-03,43510.73
3,2010-12-05,30957.28
4,2010-12-06,50907.56



Testing Dataset Preview:


,Transaction_Date,Daily_Revenue
0,2011-09-30,42791.13
1,2011-10-02,11623.58
2,2011-10-03,60423.71
3,2011-10-04,47135.98
4,2011-10-05,74470.06



Trading Calendar Preview:


,Transaction_Date,Day_of_Week,Is_Saturday,Has_Transactions,Calendar_Status
0,2010-12-01,Wednesday,False,True,Observed Trading Day
1,2010-12-02,Thursday,False,True,Observed Trading Day
2,2010-12-03,Friday,False,True,Observed Trading Day
3,2010-12-04,Saturday,True,False,Unobserved Saturday
4,2010-12-05,Sunday,False,True,Observed Trading Day


#### Defining the Forecasting Strategy

**Forecasting Objective**

Predict daily net merchandise revenue for the next seven observed trading days using historical revenue information available at the forecast origin.

**Forecasting Configuration**
- **Target Variable:** `Daily_Revenue`
- **Forecast Horizon:** 7 observed trading days
- **Forecasting Approach:** Direct multi-step forecasting
- **Prediction Strategy:** Generate seven separate horizon-specific revenue predictions from a common historical information cutoff.
- **Training Period:** December 1, 2010 – September 29, 2011
- **Testing Period:** September 30, 2011 – December 9, 2011
- **Validation Strategy:** Expanding-window time-series validation within the training period.

**Feature Engineering Principles**
- **Historical Revenue Features:** Construct lagged revenue, rolling averages, rolling variability, and trend indicators using observations available at each forecast origin.
- **Calendar Features:** Extract weekday, month, and other temporal information where the target date is known.
- **Data Leakage Prevention:** Ensure that predictions for all seven horizons use only revenue observations available before the forecast begins.
- **Chronological Integrity:** Preserve the original transaction dates and avoid shuffling or introducing artificial zero-revenue observations.
- **Reusable Artifacts:** Save feature-engineered datasets and relevant metadata for subsequent model development and evaluation.

**Forecasting Limitations**

The dataset contains irregular calendar gaps, including unobserved Saturdays and occasional non-Saturday dates. Consequently, seven observed trading days do not always correspond to a fixed number of calendar days.

The initial forecasting framework assumes that the dates of the next seven observed trading days are available when generating predictions. This assumption must be considered when interpreting model performance and assessing real-world deployment feasibility.

#### Defining the Forecasting Strategy
The Retail Sales Dynamics project adopts a **direct multi-step forecasting approach** to predict daily net merchandise revenue for the next seven observed trading days using historical information available at each forecast origin.

**Forecasting Configuration**
- **Target Variable:** `Daily_Revenue`
- **Forecast Horizon:** 7 observed trading days
- **Forecasting Approach:** Direct multi-step forecasting
- **Prediction Strategy:** Seven horizon-specific predictions generated from a common historical information cutoff.
- **Training Period:** December 1, 2010 – September 29, 2011
- **Testing Period:** September 30, 2011 – December 9, 2011
- **Validation Strategy:** Expanding-window time-series cross-validation within the training period.

**Feature Engineering Strategy**
- **Lag Features:** Capture historical revenue values from preceding observed trading days.
- **Rolling Features:** Calculate historical revenue averages, standard deviations, and other statistical summaries.
- **Calendar Features:** Extract temporal characteristics such as day of the week, month, and other relevant date-based indicators.
- **Trend Features:** Capture changes in historical revenue levels and short-term revenue momentum.
- **Leakage Prevention:** Ensure all revenue-derived features use only observations available at the forecast origin.

**Forecasting Considerations**
- **Chronological Integrity:** Preserve temporal ordering without shuffling observations or introducing artificial zero-revenue values.
- **Multi-Step Predictions:** Generate all seven forecasts without using actual revenue from the forecast horizon.
- **Trading Calendar:** Account for irregular calendar gaps, recognizing that seven observed trading days may span more than seven calendar days.
- **Testing Integrity:** Reserve the chronological testing period for final evaluation, with model selection performed exclusively on training data.
- **Reusability:** Save engineered features and supporting metadata for subsequent modeling, evaluation, and explainability.

**Forecasting Assumption**

The initial framework assumes that the dates of the next seven observed trading days are known at prediction time. This assumption must be considered when interpreting forecasting performance and assessing practical deployment.

#### Constructing Multi-Step Forecasting Targets

In [3]:
forecast_horizon = 7

revenue_series_df = (
    pd.concat([train_df, test_df], ignore_index=True)
    .sort_values("Transaction_Date")
    .reset_index(drop=True)
)

forecasting_df = revenue_series_df.rename(
    columns={
        "Transaction_Date": "Forecast_Origin",
        "Daily_Revenue": "Origin_Revenue"
    }
).copy()

for horizon in range(1, forecast_horizon + 1):
    forecasting_df[f"Target_H{horizon}"] = (
        forecasting_df["Origin_Revenue"].shift(-horizon)
    )
    forecasting_df[f"Target_Date_H{horizon}"] = (
        forecasting_df["Forecast_Origin"].shift(-horizon)
    )

target_columns = [
    f"Target_H{horizon}"
    for horizon in range(1, forecast_horizon + 1)
]

complete_forecasts_df = forecasting_df.dropna(
    subset=target_columns
).copy()

training_end_date = train_df["Transaction_Date"].max()

training_forecasts_df = complete_forecasts_df.loc[
    complete_forecasts_df["Target_Date_H7"] <= training_end_date
].copy()

testing_forecasts_df = complete_forecasts_df.loc[
    complete_forecasts_df["Forecast_Origin"] <= test_df["Transaction_Date"].max()
].copy()

testing_forecasts_df = testing_forecasts_df.loc[
    testing_forecasts_df["Target_Date_H1"] > training_end_date
].copy()

assert (
    training_forecasts_df["Target_Date_H7"] <= training_end_date
).all()

assert (
    testing_forecasts_df["Target_Date_H1"] > training_end_date
).all()

print("Multi-Step Forecasting Dataset Shape:", forecasting_df.shape)
print("Complete Seven-Step Forecasts:", len(complete_forecasts_df))
print("Training Forecast Origins:", len(training_forecasts_df))
print("Testing Forecast Origins:", len(testing_forecasts_df))

print("\nFirst Three Forecasting Observations:")
display(
    complete_forecasts_df[
        ["Forecast_Origin", "Origin_Revenue"]
        + target_columns
    ].head(3)
)

print("\nTraining and Testing Forecast Date Ranges:")
display(
    pd.DataFrame({
        "Dataset": ["Training", "Testing"],
        "First_Origin": [
            training_forecasts_df["Forecast_Origin"].min(),
            testing_forecasts_df["Forecast_Origin"].min()
        ],
        "Last_Origin": [
            training_forecasts_df["Forecast_Origin"].max(),
            testing_forecasts_df["Forecast_Origin"].max()
        ],
        "First_Target": [
            training_forecasts_df["Target_Date_H1"].min(),
            testing_forecasts_df["Target_Date_H1"].min()
        ],
        "Last_Target": [
            training_forecasts_df["Target_Date_H7"].max(),
            testing_forecasts_df["Target_Date_H7"].max()
        ]
    })
)

Multi-Step Forecasting Dataset Shape: (305, 16)
Complete Seven-Step Forecasts: 298
Training Forecast Origins: 237
Testing Forecast Origins: 55

First Three Forecasting Observations:


,Forecast_Origin,Origin_Revenue,Target_H1,Target_H2,Target_H3,Target_H4,Target_H5,Target_H6,Target_H7
0,2010-12-01,57328.60,46174.28,43510.73,30957.28,50907.56,81131.02,43375.58,48978.46
1,2010-12-02,46174.28,43510.73,30957.28,50907.56,81131.02,43375.58,48978.46,55205.10
2,2010-12-03,43510.73,30957.28,50907.56,81131.02,43375.58,48978.46,55205.10,17240.92



Training and Testing Forecast Date Ranges:


,Dataset,First_Origin,Last_Origin,First_Target,Last_Target
0,Training,2010-12-01,2011-09-21,2010-12-02,2011-09-29
1,Testing,2011-09-29,2011-12-01,2011-09-30,2011-12-09


Key Observations

- 298 complete forecasting observations were generated from 305 observed trading days, with the final seven origins excluded because their full forecast horizons are unavailable.
- 237 training forecast origins have all seven targets within the training period, while 55 testing forecast origins have all seven targets within the testing period.
- The training and testing targets are chronologically separated, preserving the September 29, 2011 cutoff.

#### Generating Historical Revenue Lag Features

In [4]:
lag_periods = [1, 2, 3, 6, 7, 12, 14, 18, 24, 30]

for lag in lag_periods:
    forecasting_df[f"Revenue_Lag_{lag}"] = (
        forecasting_df["Origin_Revenue"].shift(lag)
    )

lag_columns = [
    f"Revenue_Lag_{lag}"
    for lag in lag_periods
]

print("Generated Lag Features:", len(lag_columns))
print("Lag Periods:", lag_periods)

print("\nLag Feature Preview:")
display(
    forecasting_df[
        ["Forecast_Origin", "Origin_Revenue"]
        + lag_columns
    ].iloc[30:35]
)

print("\nMissing Values in Lag Features:")
display(
    forecasting_df[lag_columns]
    .isna()
    .sum()
    .rename("Missing_Values")
    .to_frame()
)

Generated Lag Features: 10
Lag Periods: [1, 2, 3, 6, 7, 12, 14, 18, 24, 30]

Lag Feature Preview:


,Forecast_Origin,Origin_Revenue,Revenue_Lag_1,Revenue_Lag_2,Revenue_Lag_3,Revenue_Lag_6,Revenue_Lag_7,Revenue_Lag_12,Revenue_Lag_14,Revenue_Lag_18,Revenue_Lag_24,Revenue_Lag_30
30,2011-01-16,7114.11,45977.64,19519.07,22719.81,15520.80,26058.22,4791.17,23504.21,29517.91,43375.58,57328.60
31,2011-01-17,27771.28,7114.11,45977.64,19519.07,22943.86,15520.80,10989.24,43469.76,48088.35,48978.46,46174.28
32,2011-01-18,17097.60,27771.28,7114.11,45977.64,67059.54,22943.86,14309.63,4791.17,39404.72,55205.10,43510.73
33,2011-01-19,24801.64,17097.60,27771.28,7114.11,22719.81,67059.54,31179.72,10989.24,7395.31,17240.92,30957.28
34,2011-01-20,20062.27,24801.64,17097.60,27771.28,19519.07,22719.81,38712.02,14309.63,23504.21,36425.76,50907.56



Missing Values in Lag Features:


,Missing_Values
Revenue_Lag_1,1
Revenue_Lag_2,2
Revenue_Lag_3,3
Revenue_Lag_6,6
Revenue_Lag_7,7
Revenue_Lag_12,12
Revenue_Lag_14,14
Revenue_Lag_18,18
Revenue_Lag_24,24
Revenue_Lag_30,30


Key Observations

- All 10 lag features correctly reference revenue from earlier observed trading days, with no future revenue included.
- Missing values increase with lag length, from 1 for Revenue_Lag_1 to 30 for Revenue_Lag_30. These are expected because earlier historical observations are unavailable.

#### Generating Rolling Revenue Statistics

In [5]:
rolling_windows = [3, 6, 7, 14, 30]

for window in rolling_windows:
    rolling_revenue = forecasting_df["Origin_Revenue"].rolling(
        window=window,
        min_periods=window
    )

    forecasting_df[f"Revenue_Rolling_Mean_{window}"] = (
        rolling_revenue.mean()
    )

    forecasting_df[f"Revenue_Rolling_Std_{window}"] = (
        rolling_revenue.std()
    )

    forecasting_df[f"Revenue_Rolling_Min_{window}"] = (
        rolling_revenue.min()
    )

    forecasting_df[f"Revenue_Rolling_Max_{window}"] = (
        rolling_revenue.max()
    )

rolling_columns = [
    f"Revenue_Rolling_{stat}_{window}"
    for window in rolling_windows
    for stat in ["Mean", "Std", "Min", "Max"]
]

print("Generated Rolling Features:", len(rolling_columns))
print("Rolling Windows:", rolling_windows)

print("\nRolling Feature Preview:")
display(
    forecasting_df[
        ["Forecast_Origin", "Origin_Revenue"]
        + rolling_columns
    ].iloc[30:35].round(2)
)

print("\nMissing Values in Rolling Features:")
display(
    forecasting_df[rolling_columns]
    .isna()
    .sum()
    .rename("Missing_Values")
    .to_frame()
)

Generated Rolling Features: 20
Rolling Windows: [3, 6, 7, 14, 30]

Rolling Feature Preview:


C:\Users\user\AppData\Local\Temp\ipykernel_6636\3210531151.py:39: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].iloc[30:35].round(2)


,Forecast_Origin,Origin_Revenue,Revenue_Rolling_Mean_3,Revenue_Rolling_Std_3,Revenue_Rolling_Min_3,Revenue_Rolling_Max_3,Revenue_Rolling_Mean_6,Revenue_Rolling_Std_6,Revenue_Rolling_Min_6,Revenue_Rolling_Max_6,...,Revenue_Rolling_Min_7,Revenue_Rolling_Max_7,Revenue_Rolling_Mean_14,Revenue_Rolling_Std_14,Revenue_Rolling_Min_14,Revenue_Rolling_Max_14,Revenue_Rolling_Mean_30,Revenue_Rolling_Std_30,Revenue_Rolling_Min_30,Revenue_Rolling_Max_30
30,2011-01-16,7114.11,24203.61,19850.75,7114.11,45977.64,30889.00,21727.20,7114.11,67059.54,...,7114.11,67059.54,26454.61,17356.25,4791.17,67059.54,33806.30,18237.61,4791.17,81131.02
31,2011-01-17,27771.28,26954.34,19444.64,7114.11,45977.64,31693.58,21461.91,7114.11,67059.54,...,7114.11,67059.54,25333.29,16665.79,4791.17,67059.54,33192.86,18116.35,4791.17,81131.02
32,2011-01-18,17097.60,17327.66,10330.51,7114.11,27771.28,23366.58,13033.23,7114.11,45977.64,...,7114.11,67059.54,26212.32,15801.07,7114.11,67059.54,32312.42,18239.04,4791.17,81131.02
33,2011-01-19,24801.64,23223.51,5509.06,17097.60,27771.28,23713.56,13040.28,7114.11,45977.64,...,7114.11,45977.64,27198.92,15197.11,7114.11,67059.54,32107.24,18289.36,4791.17,81131.02
34,2011-01-20,20062.27,20653.84,3885.94,17097.60,24801.64,23804.09,13007.18,7114.11,45977.64,...,7114.11,45977.64,27609.83,14896.60,7114.11,67059.54,31079.06,18061.62,4791.17,81131.02



Missing Values in Rolling Features:


,Missing_Values
Revenue_Rolling_Mean_3,2
Revenue_Rolling_Std_3,2
Revenue_Rolling_Min_3,2
Revenue_Rolling_Max_3,2
Revenue_Rolling_Mean_6,5
Revenue_Rolling_Std_6,5
Revenue_Rolling_Min_6,5
Revenue_Rolling_Max_6,5
Revenue_Rolling_Mean_7,6
Revenue_Rolling_Std_7,6


Key Observations

- 20 rolling features were generated across five windows (3, 6, 7, 14, and 30 observed trading days), capturing historical revenue levels, variability, and extremes.
- Missing values are expected, ranging from 2 for the three-day window to 29 for the 30-day window, because sufficient historical observations are unavailable at the beginning of the series.

#### Generating Revenue Trend and Momentum Features

In [6]:
forecasting_df["Revenue_Change_1"] = (
    forecasting_df["Origin_Revenue"]
    - forecasting_df["Revenue_Lag_1"]
)

forecasting_df["Revenue_Change_6"] = (
    forecasting_df["Origin_Revenue"]
    - forecasting_df["Revenue_Lag_6"]
)

forecasting_df["Revenue_Change_12"] = (
    forecasting_df["Origin_Revenue"]
    - forecasting_df["Revenue_Lag_12"]
)

forecasting_df["Revenue_Momentum_3_7"] = (
    forecasting_df["Revenue_Rolling_Mean_3"]
    - forecasting_df["Revenue_Rolling_Mean_7"]
)

forecasting_df["Revenue_Momentum_7_30"] = (
    forecasting_df["Revenue_Rolling_Mean_7"]
    - forecasting_df["Revenue_Rolling_Mean_30"]
)

forecasting_df["Revenue_Ratio_3_7"] = (
    forecasting_df["Revenue_Rolling_Mean_3"]
    / forecasting_df["Revenue_Rolling_Mean_7"].replace(0, np.nan)
)

forecasting_df["Revenue_Ratio_7_30"] = (
    forecasting_df["Revenue_Rolling_Mean_7"]
    / forecasting_df["Revenue_Rolling_Mean_30"].replace(0, np.nan)
)

trend_columns = [
    "Revenue_Change_1",
    "Revenue_Change_6",
    "Revenue_Change_12",
    "Revenue_Momentum_3_7",
    "Revenue_Momentum_7_30",
    "Revenue_Ratio_3_7",
    "Revenue_Ratio_7_30"
]

print("Generated Trend and Momentum Features:", len(trend_columns))

print("\nTrend and Momentum Feature Preview:")
display(
    forecasting_df[
        ["Forecast_Origin"] + trend_columns
    ].iloc[30:35].style.format(
        {column: "{:.2f}" for column in trend_columns}
    )
)

print("\nMissing Values in Trend and Momentum Features:")
display(
    forecasting_df[trend_columns]
    .isna()
    .sum()
    .rename("Missing_Values")
    .to_frame()
)

print("\nInfinite Values in Trend and Momentum Features:")
display(
    np.isinf(forecasting_df[trend_columns])
    .sum()
    .rename("Infinite_Values")
    .to_frame()
)

Generated Trend and Momentum Features: 7

Trend and Momentum Feature Preview:


,Forecast_Origin,Revenue_Change_1,Revenue_Change_6,Revenue_Change_12,Revenue_Momentum_3_7,Revenue_Momentum_7_30,Revenue_Ratio_3_7,Revenue_Ratio_7_30
30,2011-01-16 00:00:00,-38863.53,-8406.69,2322.94,-4489.94,-5112.75,0.84,0.85
31,2011-01-17 00:00:00,20657.17,4827.42,16782.04,-3489.27,-2749.25,0.89,0.92
32,2011-01-18 00:00:00,-10673.68,-49961.94,2787.97,-12280.77,-2703.99,0.59,0.92
33,2011-01-19 00:00:00,7704.04,2081.83,-6378.08,-348.09,-8535.64,0.99,0.73
34,2011-01-20 00:00:00,-4739.37,543.20,-18649.75,-2538.11,-7887.12,0.89,0.75



Missing Values in Trend and Momentum Features:


,Missing_Values
Revenue_Change_1,1
Revenue_Change_6,6
Revenue_Change_12,12
Revenue_Momentum_3_7,6
Revenue_Momentum_7_30,29
Revenue_Ratio_3_7,6
Revenue_Ratio_7_30,29



Infinite Values in Trend and Momentum Features:


,Infinite_Values
Revenue_Change_1,0
Revenue_Change_6,0
Revenue_Change_12,0
Revenue_Momentum_3_7,0
Revenue_Momentum_7_30,0
Revenue_Ratio_3_7,0
Revenue_Ratio_7_30,0


Key Observations

- The revenue change and momentum features capture both short-term fluctuations and differences between recent and longer-term revenue levels.
- Missing values range from 1 to 29, consistent with the historical observations required by each feature. No infinite values were detected.

#### Generating Calendar-Based Features

In [7]:
forecasting_df["Origin_Day_Of_Week"] = (
    forecasting_df["Forecast_Origin"].dt.dayofweek
)

forecasting_df["Origin_Day_Of_Month"] = (
    forecasting_df["Forecast_Origin"].dt.day
)

forecasting_df["Origin_Month"] = (
    forecasting_df["Forecast_Origin"].dt.month
)

forecasting_df["Origin_Quarter"] = (
    forecasting_df["Forecast_Origin"].dt.quarter
)

forecasting_df["Origin_Day_Of_Year"] = (
    forecasting_df["Forecast_Origin"].dt.dayofyear
)

forecasting_df["Origin_Is_Weekend"] = (
    forecasting_df["Origin_Day_Of_Week"].isin([5, 6]).astype(int)
)

origin_calendar_columns = [
    "Origin_Day_Of_Week",
    "Origin_Day_Of_Month",
    "Origin_Month",
    "Origin_Quarter",
    "Origin_Day_Of_Year",
    "Origin_Is_Weekend"
]

target_calendar_columns = []

for horizon in range(1, forecast_horizon + 1):
    target_date = forecasting_df[f"Target_Date_H{horizon}"]

    calendar_features = {
        f"Target_Day_Of_Week_H{horizon}": target_date.dt.dayofweek,
        f"Target_Month_H{horizon}": target_date.dt.month,
        f"Target_Quarter_H{horizon}": target_date.dt.quarter,
        f"Target_Is_Weekend_H{horizon}": (
            target_date.dt.dayofweek.isin([5, 6]).astype(int)
        )
    }

    for column, values in calendar_features.items():
        forecasting_df[column] = values
        target_calendar_columns.append(column)

print("Origin Calendar Features:", len(origin_calendar_columns))
print("Target Calendar Features:", len(target_calendar_columns))
print(
    "Total Calendar Features:",
    len(origin_calendar_columns) + len(target_calendar_columns)
)

print("\nOrigin Calendar Feature Preview:")
display(
    forecasting_df[
        ["Forecast_Origin"] + origin_calendar_columns
    ].head(10)
)

print("\nFirst-Horizon Target Calendar Preview:")
display(
    forecasting_df[
        ["Forecast_Origin", "Target_Date_H1"]
        + [column for column in target_calendar_columns if column.endswith("_H1")]
    ].head(10)
)

print("\nMissing Values in Origin Calendar Features:")
display(
    forecasting_df[origin_calendar_columns]
    .isna()
    .sum()
    .rename("Missing_Values")
    .to_frame()
)

Origin Calendar Features: 6
Target Calendar Features: 28
Total Calendar Features: 34

Origin Calendar Feature Preview:


C:\Users\user\AppData\Local\Temp\ipykernel_6636\1038698632.py:2: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  forecasting_df["Forecast_Origin"].dt.dayofweek
C:\Users\user\AppData\Local\Temp\ipykernel_6636\1038698632.py:18: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  forecasting_df["Forecast_Origin"].dt.dayofyear
C:\Users\user\AppData\Local\Temp\ipykernel_6636\1038698632.py:40: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  f"Target_Day_Of_Week_H{horizon}": target_date.dt.dayofweek,
C:\Users\user\AppData\Local\Temp\ipykernel_6636\1038698632.py:44: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  target_date.dt.dayofweek.isin([5, 6]).astype(int)
C:\Users\user\AppData

,Forecast_Origin,Origin_Day_Of_Week,Origin_Day_Of_Month,Origin_Month,Origin_Quarter,Origin_Day_Of_Year,Origin_Is_Weekend
0,2010-12-01,2,1,12,4,335,0
1,2010-12-02,3,2,12,4,336,0
2,2010-12-03,4,3,12,4,337,0
3,2010-12-05,6,5,12,4,339,1
4,2010-12-06,0,6,12,4,340,0
5,2010-12-07,1,7,12,4,341,0
6,2010-12-08,2,8,12,4,342,0
7,2010-12-09,3,9,12,4,343,0
8,2010-12-10,4,10,12,4,344,0
9,2010-12-12,6,12,12,4,346,1



First-Horizon Target Calendar Preview:


,Forecast_Origin,Target_Date_H1,Target_Day_Of_Week_H1,Target_Month_H1,Target_Quarter_H1,Target_Is_Weekend_H1
0,2010-12-01,2010-12-02,3.0,12.0,4.0,0
1,2010-12-02,2010-12-03,4.0,12.0,4.0,0
2,2010-12-03,2010-12-05,6.0,12.0,4.0,1
3,2010-12-05,2010-12-06,0.0,12.0,4.0,0
4,2010-12-06,2010-12-07,1.0,12.0,4.0,0
5,2010-12-07,2010-12-08,2.0,12.0,4.0,0
6,2010-12-08,2010-12-09,3.0,12.0,4.0,0
7,2010-12-09,2010-12-10,4.0,12.0,4.0,0
8,2010-12-10,2010-12-12,6.0,12.0,4.0,1
9,2010-12-12,2010-12-13,0.0,12.0,4.0,0



Missing Values in Origin Calendar Features:


,Missing_Values
Origin_Day_Of_Week,0
Origin_Day_Of_Month,0
Origin_Month,0
Origin_Quarter,0
Origin_Day_Of_Year,0
Origin_Is_Weekend,0


Key Observations

- 34 calendar features were generated: 6 describing the forecast origin and 28 describing the seven future forecast horizons.
- All origin calendar features contain zero missing values, and the previews confirm that weekdays, months, quarters, and weekend indicators were extracted correctly.

#### Updating Calendar Feature Compatibility

In [8]:
forecasting_df["Origin_Day_Of_Week"] = (
    forecasting_df["Forecast_Origin"].dt.day_of_week
)

forecasting_df["Origin_Day_Of_Year"] = (
    forecasting_df["Forecast_Origin"].dt.day_of_year
)

for horizon in range(1, forecast_horizon + 1):
    target_date = forecasting_df[f"Target_Date_H{horizon}"]

    forecasting_df[f"Target_Day_Of_Week_H{horizon}"] = (
        target_date.dt.day_of_week
    )

    forecasting_df[f"Target_Is_Weekend_H{horizon}"] = (
        target_date.dt.day_of_week.isin([5, 6]).astype(int)
    )

print("Calendar Features Updated Successfully.")

print("\nCalendar Feature Summary:")
print("Origin Calendar Features:", len(origin_calendar_columns))
print("Target Calendar Features:", len(target_calendar_columns))
print("Total Calendar Features:", len(origin_calendar_columns) + len(target_calendar_columns))

print("\nMissing Values in Origin Calendar Features:")
display(
    forecasting_df[origin_calendar_columns]
    .isna()
    .sum()
    .rename("Missing_Values")
    .to_frame()
)

Calendar Features Updated Successfully.

Calendar Feature Summary:
Origin Calendar Features: 6
Target Calendar Features: 28
Total Calendar Features: 34

Missing Values in Origin Calendar Features:


,Missing_Values
Origin_Day_Of_Week,0
Origin_Day_Of_Month,0
Origin_Month,0
Origin_Quarter,0
Origin_Day_Of_Year,0
Origin_Is_Weekend,0


#### Validating Engineered Features

In [9]:
feature_groups = {
    "Historical Lag Features": lag_columns,
    "Rolling Revenue Features": rolling_columns,
    "Trend and Momentum Features": trend_columns,
    "Origin Calendar Features": origin_calendar_columns,
    "Target Calendar Features": target_calendar_columns
}

all_feature_columns = [
    column
    for columns in feature_groups.values()
    for column in columns
]

feature_summary = pd.DataFrame({
    "Feature_Group": list(feature_groups.keys()),
    "Feature_Count": [
        len(columns) for columns in feature_groups.values()
    ],
    "Missing_Values": [
        forecasting_df[columns].isna().sum().sum()
        for columns in feature_groups.values()
    ]
})

print("Feature Engineering Summary:")
display(feature_summary)

print("\nTotal Engineered Features:", len(all_feature_columns))
print("Unique Feature Names:", len(set(all_feature_columns)))
print("Forecasting Dataset Shape:", forecasting_df.shape)

print("\nFeature Validation:")
assert len(all_feature_columns) == len(set(all_feature_columns))
assert set(all_feature_columns).issubset(forecasting_df.columns)

numeric_features = forecasting_df[all_feature_columns].select_dtypes(
    include="number"
)

infinite_count = np.isinf(numeric_features).sum().sum()

print("Infinite Feature Values:", infinite_count)
assert infinite_count == 0

print("\nMissing Values by Feature:")
missing_features = (
    forecasting_df[all_feature_columns]
    .isna()
    .sum()
)

display(
    missing_features[missing_features > 0]
    .rename("Missing_Values")
    .to_frame()
)

print("\nHistorical Feature Availability:")
historical_feature_columns = (
    lag_columns + rolling_columns + trend_columns
)

first_complete_index = forecasting_df[
    historical_feature_columns
].dropna().index.min()

print("First Complete Row Index:", first_complete_index)
print(
    "First Complete Forecast Origin:",
    forecasting_df.loc[first_complete_index, "Forecast_Origin"]
)

print("\nFeature Validation Completed Successfully.")

Feature Engineering Summary:


,Feature_Group,Feature_Count,Missing_Values
0,Historical Lag Features,10,117
1,Rolling Revenue Features,20,220
2,Trend and Momentum Features,7,89
3,Origin Calendar Features,6,0
4,Target Calendar Features,28,84



Total Engineered Features: 71
Unique Feature Names: 71
Forecasting Dataset Shape: (305, 87)

Feature Validation:
Infinite Feature Values: 0

Missing Values by Feature:


,Missing_Values
Revenue_Lag_1,1
Revenue_Lag_2,2
Revenue_Lag_3,3
Revenue_Lag_6,6
Revenue_Lag_7,7
Revenue_Lag_12,12
Revenue_Lag_14,14
Revenue_Lag_18,18
Revenue_Lag_24,24
Revenue_Lag_30,30



Historical Feature Availability:
First Complete Row Index: 30
First Complete Forecast Origin: 2011-01-16 00:00:00

Feature Validation Completed Successfully.


Key Observations
- 71 engineered features were validated across five feature groups, with no duplicate feature names or infinite values.
- Missing values are confined to the initial historical lookback period and unavailable future target dates. The first observation with complete historical features is January 16, 2011.
- The forecasting dataset contains 305 observations and 87 columns, including the forecast origins, targets, and engineered features.

#### Preparing Final Forecasting Datasets

In [10]:
historical_feature_columns = (
    lag_columns + rolling_columns + trend_columns
)

required_columns = (
    historical_feature_columns
    + origin_calendar_columns
    + target_calendar_columns
    + target_columns
)

modeling_df = (
    forecasting_df
    .dropna(subset=required_columns)
    .copy()
    .reset_index(drop=True)
)

train_features_df = modeling_df.loc[
    modeling_df["Target_Date_H7"] <= training_end_date
].copy().reset_index(drop=True)

test_features_df = modeling_df.loc[
    modeling_df["Target_Date_H1"] > training_end_date
].copy().reset_index(drop=True)

assert train_features_df["Target_Date_H7"].le(training_end_date).all()
assert test_features_df["Target_Date_H1"].gt(training_end_date).all()

assert not train_features_df[required_columns].isna().any().any()
assert not test_features_df[required_columns].isna().any().any()

print("Complete Modeling Dataset:", modeling_df.shape)
print("Training Feature Dataset:", train_features_df.shape)
print("Testing Feature Dataset:", test_features_df.shape)

print("\nForecasting Dataset Summary:")
display(
    pd.DataFrame({
        "Dataset": ["Training", "Testing"],
        "Observations": [
            len(train_features_df),
            len(test_features_df)
        ],
        "First_Origin": [
            train_features_df["Forecast_Origin"].min(),
            test_features_df["Forecast_Origin"].min()
        ],
        "Last_Origin": [
            train_features_df["Forecast_Origin"].max(),
            test_features_df["Forecast_Origin"].max()
        ],
        "First_Target": [
            train_features_df["Target_Date_H1"].min(),
            test_features_df["Target_Date_H1"].min()
        ],
        "Last_Target": [
            train_features_df["Target_Date_H7"].max(),
            test_features_df["Target_Date_H7"].max()
        ]
    })
)

print("\nMissing Values:")
print("Training:", train_features_df[required_columns].isna().sum().sum())
print("Testing:", test_features_df[required_columns].isna().sum().sum())

Complete Modeling Dataset: (268, 87)
Training Feature Dataset: (207, 87)
Testing Feature Dataset: (55, 87)

Forecasting Dataset Summary:


,Dataset,Observations,First_Origin,Last_Origin,First_Target,Last_Target
0,Training,207,2011-01-16,2011-09-21,2011-01-17,2011-09-29
1,Testing,55,2011-09-29,2011-12-01,2011-09-30,2011-12-09



Missing Values:
Training: 0
Testing: 0


**Key Observations**
- **Modeling Dataset:** After removing incomplete historical observations and forecast horizons, 268 complete forecasting observations remain, each containing 87 columns.
- **Chronological Separation:** The training dataset contains 207 forecast origins (January 16 – September 21, 2011), while the testing dataset contains 55 forecast origins (September 29 – December 1, 2011). All training targets remain within the training period, and all testing targets occur after the training cutoff.
- **Data Completeness:** Both datasets contain zero missing values across the required features and targets, making them ready for horizon-specific modeling.

#### Organizing Horizon-Specific Predictor Features

In [11]:
shared_feature_columns = (
    ["Origin_Revenue"]
    + historical_feature_columns
    + origin_calendar_columns
)

horizon_feature_columns = {}
horizon_target_columns = {}

for horizon in range(1, forecast_horizon + 1):
    horizon_calendar_features = [
        f"Target_Day_Of_Week_H{horizon}",
        f"Target_Month_H{horizon}",
        f"Target_Quarter_H{horizon}",
        f"Target_Is_Weekend_H{horizon}"
    ]

    horizon_feature_columns[horizon] = (
        shared_feature_columns + horizon_calendar_features
    )

    horizon_target_columns[horizon] = f"Target_H{horizon}"

feature_configuration = pd.DataFrame([
    {
        "Horizon": horizon,
        "Historical_and_Origin_Features": len(shared_feature_columns),
        "Target_Calendar_Features": 4,
        "Total_Predictors": len(horizon_feature_columns[horizon]),
        "Target": horizon_target_columns[horizon]
    }
    for horizon in range(1, forecast_horizon + 1)
])

print("Horizon-Specific Feature Configuration:")
display(feature_configuration)

print("\nPredictor Validation:")

for horizon in range(1, forecast_horizon + 1):
    predictors = horizon_feature_columns[horizon]
    target = horizon_target_columns[horizon]

    assert len(predictors) == len(set(predictors))
    assert set(predictors).issubset(train_features_df.columns)
    assert set(predictors).issubset(test_features_df.columns)
    assert target not in predictors
    assert not any(column.startswith("Target_H") for column in predictors)

    assert not train_features_df[predictors].isna().any().any()
    assert not test_features_df[predictors].isna().any().any()

print("All seven horizon configurations validated successfully.")

print("\nExample Predictors for Horizon 1:")
print(horizon_feature_columns[1])

print("\nExample Predictors for Horizon 7:")
print(horizon_feature_columns[7])

Horizon-Specific Feature Configuration:


,Horizon,Historical_and_Origin_Features,Target_Calendar_Features,Total_Predictors,Target
0,1,44,4,48,Target_H1
1,2,44,4,48,Target_H2
2,3,44,4,48,Target_H3
3,4,44,4,48,Target_H4
4,5,44,4,48,Target_H5
5,6,44,4,48,Target_H6
6,7,44,4,48,Target_H7



Predictor Validation:
All seven horizon configurations validated successfully.

Example Predictors for Horizon 1:
['Origin_Revenue', 'Revenue_Lag_1', 'Revenue_Lag_2', 'Revenue_Lag_3', 'Revenue_Lag_6', 'Revenue_Lag_7', 'Revenue_Lag_12', 'Revenue_Lag_14', 'Revenue_Lag_18', 'Revenue_Lag_24', 'Revenue_Lag_30', 'Revenue_Rolling_Mean_3', 'Revenue_Rolling_Std_3', 'Revenue_Rolling_Min_3', 'Revenue_Rolling_Max_3', 'Revenue_Rolling_Mean_6', 'Revenue_Rolling_Std_6', 'Revenue_Rolling_Min_6', 'Revenue_Rolling_Max_6', 'Revenue_Rolling_Mean_7', 'Revenue_Rolling_Std_7', 'Revenue_Rolling_Min_7', 'Revenue_Rolling_Max_7', 'Revenue_Rolling_Mean_14', 'Revenue_Rolling_Std_14', 'Revenue_Rolling_Min_14', 'Revenue_Rolling_Max_14', 'Revenue_Rolling_Mean_30', 'Revenue_Rolling_Std_30', 'Revenue_Rolling_Min_30', 'Revenue_Rolling_Max_30', 'Revenue_Change_1', 'Revenue_Change_6', 'Revenue_Change_12', 'Revenue_Momentum_3_7', 'Revenue_Momentum_7_30', 'Revenue_Ratio_3_7', 'Revenue_Ratio_7_30', 'Origin_Day_Of_Week', 'Or

**Key Observations**
- **Horizon-Specific Configuration:** Seven forecasting configurations were established, each containing 48 predictors: 44 shared historical and origin features and 4 horizon-specific calendar features.
- **Feature Validation:** All configurations passed validation, with unique predictor names, no missing predictor values, and no future revenue targets included as model inputs.

#### Saving Feature-Engineered Datasets

In [12]:
import json

artifacts_dir = Path("../artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

train_path = data_dir / "forecast_features_train.parquet"
test_path = data_dir / "forecast_features_test.parquet"
config_path = artifacts_dir / "forecast_feature_config.json"

train_features_df.to_parquet(train_path, index=False)
test_features_df.to_parquet(test_path, index=False)

feature_config = {
    "forecast_horizon": forecast_horizon,
    "forecasting_approach": "direct_multi_step",
    "horizon_unit": "observed_trading_days",
    "training_end_date": str(training_end_date.date()),
    "future_trading_dates_assumed_known": True,
    "shared_features": shared_feature_columns,
    "horizon_features": {
        str(horizon): horizon_feature_columns[horizon]
        for horizon in range(1, forecast_horizon + 1)
    },
    "horizon_targets": {
        str(horizon): horizon_target_columns[horizon]
        for horizon in range(1, forecast_horizon + 1)
    }
}

with open(config_path, "w", encoding="utf-8") as file:
    json.dump(feature_config, file, indent=4)

saved_train_df = pd.read_parquet(train_path)
saved_test_df = pd.read_parquet(test_path)

with open(config_path, "r", encoding="utf-8") as file:
    saved_config = json.load(file)

assert saved_train_df.equals(train_features_df)
assert saved_test_df.equals(test_features_df)
assert saved_config["forecast_horizon"] == 7
assert len(saved_config["horizon_features"]) == 7

print("Feature-Engineered Artifacts Saved Successfully.")

print("\nSaved Files:")
print(train_path)
print(test_path)
print(config_path)

print("\nVerified Dataset Shapes:")
print("Training:", saved_train_df.shape)
print("Testing:", saved_test_df.shape)

print("\nVerified Forecasting Configuration:")
print("Forecast Horizon:", saved_config["forecast_horizon"])
print("Forecasting Approach:", saved_config["forecasting_approach"])
print("Horizon Configurations:", len(saved_config["horizon_features"]))

Feature-Engineered Artifacts Saved Successfully.

Saved Files:
..\data\processed\forecast_features_train.parquet
..\data\processed\forecast_features_test.parquet
..\artifacts\forecast_feature_config.json

Verified Dataset Shapes:
Training: (207, 87)
Testing: (55, 87)

Verified Forecasting Configuration:
Forecast Horizon: 7
Forecasting Approach: direct_multi_step
Horizon Configurations: 7
